This notebook runs on KAGGLE, not Colab, due to Colab GPU quota limits.
After training completes, manually transfer cnn_best.pt, cnn_split_indices.json,
and figures back to Google Drive DATA_ROOT before running Chunk 8, which
still runs on Colab and expects those files at their standard DATA_ROOT paths.

## Setup and Dependency Installation


In [1]:
!pip install wandb tqdm -q

## Repository Setup


In [2]:
import os
REPO_ROOT = '/kaggle/working/antenna-gnn'
if not os.path.exists(REPO_ROOT):
    !git clone https://github.com/asparagusD/antenna_gnn.git {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull --quiet
import sys
sys.path.insert(0, f'{REPO_ROOT}/src')
print(f'Repo ready at {REPO_ROOT}')

Cloning into '/kaggle/working/antenna-gnn'...
Username for 'https://github.com': ^C
Repo ready at /kaggle/working/antenna-gnn


## Drive Mount and Path Configuration (Kaggle)


In [5]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for file in files:
        print(os.path.join(root, file))


/kaggle/input/datasets/naufelaurnob/antenna-cnn-dataset-25x25/cnn_dataset_25x25_full.pt


In [6]:
import os
KAGGLE_INPUT = '/kaggle/input/datasets/naufelaurnob/antenna-cnn-dataset-25x25'
WORKING_DIR = '/kaggle/working'
os.makedirs(f'{WORKING_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{WORKING_DIR}/figures', exist_ok=True)
os.makedirs(f'{WORKING_DIR}/splits', exist_ok=True)

# Verify the cached dataset file is visible
cache_path = f'{KAGGLE_INPUT}/cnn_dataset_25x25_full.pt'
assert os.path.exists(cache_path), (
    f'Cached dataset not found at {cache_path}. Confirm the Kaggle Dataset '
    f'is attached to this notebook via the Add Data panel, and that the '
    f'filename matches exactly.'
)
print(f'Found cached dataset at {cache_path}')

Found cached dataset at /kaggle/input/datasets/naufelaurnob/antenna-cnn-dataset-25x25/cnn_dataset_25x25_full.pt


## Section 2: Load dataset (Fast path)


In [8]:
import torch
print('Loading cached dataset from Kaggle input...')
cached = torch.load(cache_path, weights_only=False)
X, y, is_functioning = cached['X'], cached['y'], cached['is_functioning']
print(f'Loaded: X {X.shape}, y {y.shape}')
print(f'Functioning: {is_functioning.sum()} / {len(is_functioning)} '
      f'({100*is_functioning.mean():.1f}%)')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
assert device.type == 'cuda', 'Enable GPU: Settings -> Accelerator -> GPU T4 x2'
print(f'Using device: {device}')

Loading cached dataset from Kaggle input...
Loaded: X torch.Size([99833, 1, 25, 25]), y torch.Size([99833, 201])
Functioning: 54853 / 99833 (54.9%)
Using device: cuda


## Stratified split


In [9]:
from sklearn.model_selection import StratifiedShuffleSplit
import json
import numpy as np

y_func = is_functioning
sss_test = StratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_val_idx, test_idx = next(sss_test.split(np.zeros(len(y_func)), y_func))

sss_val = StratifiedShuffleSplit(n_splits=1, test_size=1/9, random_state=42)
train_idx_rel, val_idx_rel = next(sss_val.split(np.zeros(len(train_val_idx)), y_func[train_val_idx]))

train_idx = train_val_idx[train_idx_rel]
val_idx = train_val_idx[val_idx_rel]

with open(f'{WORKING_DIR}/splits/cnn_split_indices.json', 'w') as f_out:
    json.dump({
        'train': train_idx.tolist(),
        'val': val_idx.tolist(),
        'test': test_idx.tolist()
    }, f_out)

print(f"Total samples: {len(X)}")
print(f"Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}")
print(f"Functioning % in Train: {100 * np.mean(y_func[train_idx]):.2f}%")
print(f"Functioning % in Val:   {100 * np.mean(y_func[val_idx]):.2f}%")
print(f"Functioning % in Test:  {100 * np.mean(y_func[test_idx]):.2f}%")

Total samples: 99833
Train: 79865, Val: 9984, Test: 9984
Functioning % in Train: 54.94%
Functioning % in Val:   54.95%
Functioning % in Test:  54.95%


## Section 3: CNN model (AntennaCNN)


In [10]:
import torch.nn as nn

class AntennaCNN(nn.Module):
    """
    Adapted from Gupta et al. IEEE TAP 2023, Fig. 4.
    Original: 12x12 input, 81 output points.
    Adapted: NxN input via AdaptiveAvgPool2d, 201 output points (1-4 GHz).
    """
    def __init__(self):
        super().__init__()
        
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2),
            nn.Conv2d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(0.2)
        )
        
        layers2 = []
        layers2.extend([nn.Conv2d(64, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2)])
        for _ in range(4):
            layers2.extend([nn.Conv2d(128, 128, kernel_size=3, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2)])
        self.block2 = nn.Sequential(*layers2)
        
        layers3 = []
        layers3.extend([nn.Conv2d(128, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2)])
        for _ in range(7):
            layers3.extend([nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.LeakyReLU(0.2)])
        self.block3 = nn.Sequential(*layers3)
        
        self.readout = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten()
        )
        
        self.fc = nn.Sequential(
            nn.Linear(256, 1000),
            nn.BatchNorm1d(1000),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(1000, 500),
            nn.BatchNorm1d(500),
            nn.LeakyReLU(0.2),
            nn.Dropout(0.4),
            nn.Linear(500, 201)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.readout(x)
        x = self.fc(x)
        return x

model = AntennaCNN().to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

Total parameters: 6,163,473


## Section 4: Training loop
Kaggle sessions have a max runtime (12 hours for GPU sessions). If checkpoint exists at `WORKING_DIR/checkpoints/cnn_best.pt` at notebook start, load and resume — but note this ONLY helps within a single Kaggle session with 'Save & Run All' re-runs, since `/kaggle/working` does not persist between separate sessions unless the notebook output is committed.


In [11]:
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(TensorDataset(X[train_idx], y[train_idx]), batch_size=256, shuffle=True)
val_loader = DataLoader(TensorDataset(X[val_idx], y[val_idx]), batch_size=256)

optimizer = torch.optim.NAdam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

def save_checkpoint(model, optimizer, epoch, val_loss, path):
    torch.save({
        'epoch': epoch,
        'model_state': model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'val_loss': val_loss,
    }, path)

CKPT_PATH = f'{WORKING_DIR}/checkpoints/cnn_best.pt'
start_epoch = 0
best_val_loss = float('inf')

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_epoch = ckpt['epoch'] + 1
    best_val_loss = ckpt['val_loss']
    print(f'Resumed from epoch {start_epoch}, val_loss={best_val_loss:.6f}')
else:
    print('Starting fresh training')

train_losses = []
val_losses = []
epochs = 40

for epoch in range(start_epoch, epochs):
    model.train()
    total_train_loss = 0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        out = model(batch_X)
        loss = criterion(out, batch_y)
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item() * batch_X.size(0)
        del batch_X, batch_y
        torch.cuda.empty_cache()
        
    avg_train_loss = total_train_loss / len(train_idx)
    
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_X, batch_y in val_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)
            out = model(batch_X)
            loss = criterion(out, batch_y)
            total_val_loss += loss.item() * batch_X.size(0)
            del batch_X, batch_y
            torch.cuda.empty_cache()
            
    avg_val_loss = total_val_loss / len(val_idx)
    
    # Store for plotting (if resuming, this list only tracks the new epochs)
    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_checkpoint(model, optimizer, epoch, best_val_loss, CKPT_PATH)
        
    if (epoch + 1) % 10 == 0 or epoch == start_epoch:
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

Starting fresh training
Epoch 001 | Train Loss: 3.3661 | Val Loss: 2.5488
Epoch 010 | Train Loss: 1.7726 | Val Loss: 2.4942
Epoch 020 | Train Loss: 1.5437 | Val Loss: 3.4972
Epoch 030 | Train Loss: 1.4030 | Val Loss: 3.1577
Epoch 040 | Train Loss: 1.2957 | Val Loss: 1.7494


## Section 5: Training curves


In [12]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

if train_losses:
    plt.figure(figsize=(10, 6))
    plt.plot(range(start_epoch, epochs), train_losses, label='Train Loss')
    plt.plot(range(start_epoch, epochs), val_losses, label='Val Loss')
    plt.title('CNN Baseline Training Curves (Gupta et al. 2023 architecture)')
    plt.xlabel('Epoch')
    plt.ylabel('MSE Loss')
    plt.legend()
    plt.grid(True)
    plt.savefig(f'{WORKING_DIR}/figures/cnn_training_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Final Train Loss: {train_losses[-1]:.4f}")
    print(f"Final Val Loss: {val_losses[-1]:.4f}")
else:
    print("Model was already fully trained.")

Final Train Loss: 1.2957
Final Val Loss: 1.7494


## Section 6: Evaluation


In [13]:
from scipy.signal import find_peaks

def extract_resonant_freq(s11_db, freq_axis_ghz, threshold_db=-10):
    inverted = -s11_db
    peaks, props = find_peaks(inverted, height=-threshold_db, distance=5)
    if len(peaks) == 0:
        return None
    deepest = peaks[np.argmax(inverted[peaks])]
    return freq_axis_ghz[deepest]

ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model_state'])
model.eval()

freq_axis = np.linspace(1.0, 4.0, 201)
test_loader = DataLoader(TensorDataset(X[test_idx], y[test_idx]), batch_size=256)

all_preds = []
all_trues = []

with torch.no_grad():
    for batch_X, batch_y in test_loader:
        out = model(batch_X.to(device))
        all_preds.append(out.cpu())
        all_trues.append(batch_y)

all_preds = torch.cat(all_preds).numpy()
all_trues = torch.cat(all_trues).numpy()

mae_s11 = np.mean(np.abs(all_preds - all_trues))

mae_freqs = []
for i in range(len(all_trues)):
    true_res = extract_resonant_freq(all_trues[i], freq_axis)
    if true_res is not None:
        pred_res = extract_resonant_freq(all_preds[i], freq_axis)
        if pred_res is not None:
            mae_freqs.append(abs(true_res - pred_res))

mae_freq = np.mean(mae_freqs) if len(mae_freqs) > 0 else float('nan')

print("--------------------------------------------------")
print("=== Test Set Evaluation ===")
print(f"MAE S11 Spectrum : {mae_s11:.4f} dB")
print(f"MAE Resonant Freq: {mae_freq:.4f} GHz")
print("--------------------------------------------------")

--------------------------------------------------
=== Test Set Evaluation ===
MAE S11 Spectrum : 0.4036 dB
MAE Resonant Freq: 0.0215 GHz
--------------------------------------------------


## Section 7: Prediction plots


In [14]:
import random

functioning_idx = []
non_functioning_idx = []

for i in range(len(all_trues)):
    if extract_resonant_freq(all_trues[i], freq_axis) is not None:
        functioning_idx.append(i)
    else:
        non_functioning_idx.append(i)

random.seed(42)
sampled_func = random.sample(functioning_idx, min(3, len(functioning_idx)))
sampled_nonfunc = random.sample(non_functioning_idx, min(3, len(non_functioning_idx)))
plot_indices = sampled_func + sampled_nonfunc

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for i, idx in enumerate(plot_indices):
    ax = axes[i]
    ax.plot(freq_axis, all_trues[idx], 'b-', label='True')
    ax.plot(freq_axis, all_preds[idx], 'r--', label='Predicted')
    ax.axhline(-10, color='gray', linestyle=':')
    title = 'Functioning' if i < len(sampled_func) else 'Non-functioning'
    ax.set_title(title)
    ax.set_xlabel('Frequency (GHz)')
    ax.set_ylabel('S11 (dB)')
    if i == 0:
        ax.legend()

plt.tight_layout()
plt.savefig(f'{WORKING_DIR}/figures/cnn_predictions.png', dpi=300, bbox_inches='tight')
plt.show()

## Section 8: Permutation sensitivity demonstration


In [15]:
random.seed(42)
perm_test_indices = random.sample(range(len(test_idx)), 20)
test_X = X[test_idx]

std_list = []
example_preds = []

model.eval()
with torch.no_grad():
    for i, idx in enumerate(perm_test_indices):
        orig = test_X[idx] # (1, 25, 25)
        orig_t = torch.transpose(orig, 1, 2)
        
        variants = [
            orig,
            orig_t,
            torch.flip(orig, [2]),
            torch.flip(orig, [1]),
            torch.flip(orig_t, [2]),
            torch.flip(orig_t, [1]),
            torch.flip(orig, [1, 2]),
            torch.flip(orig_t, [1, 2])
        ]
        
        batch_var = torch.stack(variants).to(device) # (8, 1, 25, 25)
        preds = model(batch_var).cpu().numpy() # (8, 201)
        
        std_list.append(np.std(preds, axis=0)) # (201,) std across variants
        
        if i == 0:
            example_preds = preds

mean_sensitivity = np.mean(std_list)
print(f"CNN permutation sensitivity: {mean_sensitivity:.2f} dB")

plt.figure(figsize=(8, 6))
for i in range(8):
    plt.plot(freq_axis, example_preds[i], label=f'Variant {i+1}')
plt.axhline(-10, color='gray', linestyle=':')
plt.title('CNN Predictions on 8 Spatial Variants of 1 Antenna')
plt.xlabel('Frequency (GHz)')
plt.ylabel('S11 (dB)')
plt.legend()
plt.savefig(f'{WORKING_DIR}/figures/cnn_permutation_sensitivity.png', dpi=300, bbox_inches='tight')
plt.show()

CNN permutation sensitivity: 0.28 dB


## Packaging Output for Google Drive


In [16]:
!zip -r /kaggle/working/cnn_chunk3_outputs.zip \
    /kaggle/working/checkpoints \
    /kaggle/working/figures \
    /kaggle/working/splits

print('Training complete. Download cnn_chunk3_outputs.zip from the Kaggle')
print('Output panel, then manually upload cnn_best.pt to')
print('Google Drive at DATA_ROOT/checkpoints/cnn_best.pt, cnn_split_indices.json')
print('to DATA_ROOT/splits/, and figures to DATA_ROOT/figures/ so Chunk 8 can')
print('find them.')

  adding: kaggle/working/checkpoints/ (stored 0%)
  adding: kaggle/working/checkpoints/cnn_best.pt (deflated 8%)
  adding: kaggle/working/figures/ (stored 0%)
  adding: kaggle/working/figures/cnn_permutation_sensitivity.png (deflated 12%)
  adding: kaggle/working/figures/cnn_training_curves.png (deflated 11%)
  adding: kaggle/working/figures/cnn_predictions.png (deflated 17%)
  adding: kaggle/working/splits/ (stored 0%)
  adding: kaggle/working/splits/cnn_split_indices.json (deflated 58%)
Training complete. Download cnn_chunk3_outputs.zip from the Kaggle
Output panel, then manually upload cnn_best.pt to
Google Drive at DATA_ROOT/checkpoints/cnn_best.pt, cnn_split_indices.json
to DATA_ROOT/splits/, and figures to DATA_ROOT/figures/ so Chunk 8 can
find them.
